# Test Dataset Loading - MSLR ICCV 2025

Notebook untuk menguji dan memverifikasi berapa banyak data train, dev, dan test yang berhasil di-load dari dataset.

In [1]:
import os
import sys
import json
import glob
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

# Add project root to path
sys.path.insert(0, '/content/MSLR_ICCV2025')
sys.path.insert(0, '.')

# PyTorch imports
import torch
from torch.utils.data import DataLoader

print("✓ All imports successful")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Set base paths
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "datasets" / "mslr2025"
CSV_PATH = DATA_ROOT / "preprocess" / "mslr2025" / "SD" if (PROJECT_ROOT / "preprocess").exists() else PROJECT_ROOT / "preprocess" / "mslr2025" / "SD"

# Try to find CSV files
if not CSV_PATH.exists():
    CSV_PATH = PROJECT_ROOT / "preprocess" / "mslr2025" / "SD"

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Data Root: {DATA_ROOT}")
print(f"📁 CSV Path: {CSV_PATH}")
print(f"✓ Paths configured\n")

# Configuration
config = {
    'setting': 'sd',
    'datatype': 'skeleton',
    'dataset': 'bisindo',
}

print(f"Config: {config}")

In [ ]:
def load_csv_data(csv_file):
    """Load data from CSV file and return count"""
    try:
        df = pd.read_csv(csv_file, sep='|')
        # Skip header row if exists
        if len(df) > 0:
            return len(df), df
        else:
            return 0, None
    except Exception as e:
        print(f"❌ Error loading {csv_file}: {e}")
        return 0, None

# Define CSV file paths
train_csv = CSV_PATH / "train.csv"
dev_csv = CSV_PATH / "dev.csv"
test_csv = CSV_PATH / "test.csv"

print("Loading data from CSV files...\n")

# Load each split
train_count, train_df = load_csv_data(train_csv)
dev_count, dev_df = load_csv_data(dev_csv)
test_count, test_df = load_csv_data(test_csv)

# Display results
print("=" * 60)
print("📊 DATASET LOADING RESULTS")
print("=" * 60)
print(f"Train samples: {train_count}")
print(f"Dev samples:   {dev_count}")
print(f"Test samples:  {test_count}")
print(f"Total samples: {train_count + dev_count + test_count}")
print("=" * 60 + "\n")

In [ ]:
# Display sample data from each split
print("📝 SAMPLE DATA (First 5 samples per split)\n")

if train_df is not None and len(train_df) > 0:
    print("TRAIN samples:")
    print(train_df.head())
    print()

if dev_df is not None and len(dev_df) > 0:
    print("DEV samples:")
    print(dev_df.head())
    print()

if test_df is not None and len(test_df) > 0:
    print("TEST samples:")
    print(test_df.head())
    print()

In [ ]:
try:
    import datasets
    from datasets import SkeletonFeeder
    import yaml
    
    print("Loading dataset config...\n")
    
    # Load config
    config_path = PROJECT_ROOT / "configs" / "dataset_configs" / "bisindo.yaml"
    with open(config_path, 'r') as f:
        dataset_info = yaml.load(f, Loader=yaml.FullLoader)
    
    print(f"✓ Config loaded from {config_path}")
    print(f"  Dataset root: {dataset_info['dataset_root']}")
    print(f"  Dict path: {dataset_info['dict_path']}\n")
    
    # Load gloss dictionary
    dict_path = dataset_info['dict_path']
    with open(dict_path, 'r') as f:
        gloss_dict = json.load(f)
    
    print(f"✓ Gloss dictionary loaded ({len(gloss_dict['gloss2id'])} glosses)\n")
    
    # Create dataset for each split
    print("=" * 60)
    print("🔄 LOADING WITH SkeletonFeeder")
    print("=" * 60 + "\n")
    
    datasets_loaded = {}
    g2i_dict = {k: v['index'] for k, v in gloss_dict['gloss2id'].items()}
    
    for split in ['train', 'dev', 'test']:
        print(f"Loading {split}...")
        feeder_args = {
            'mode': split,
            'transform_mode': (split == 'train'),
            'dataset': 'bisindo',
            'setting': 'sd',
            'datatype': 'skeleton',
            'split': [25, 46, 67, 86],
            'norm_point': [0, 25, 46, 67],
            'used_part': ['body', 'hand21', 'mouth_8'],
            'augmentation_types': [],
            'normalization_types': ['spatial'],
            'downsampling': False,  # Disable downsampling for testing
        }
        
        dataset = SkeletonFeeder(gloss_dict=g2i_dict, **feeder_args)
        datasets_loaded[split] = dataset
        print(f"  ✓ {split}: {len(dataset)} samples\n")
    
    print("=" * 60)
    print("✅ SKELETON FEEDER RESULTS")
    print("=" * 60)
    print(f"Train (SkeletonFeeder): {len(datasets_loaded['train'])}")
    print(f"Dev (SkeletonFeeder):   {len(datasets_loaded['dev'])}")
    print(f"Test (SkeletonFeeder):  {len(datasets_loaded['test'])}")
    print(f"Total (SkeletonFeeder): {sum(len(datasets_loaded[s]) for s in ['train', 'dev', 'test'])}")
    print("=" * 60 + "\n")
    
except Exception as e:
    print(f"❌ Error loading with SkeletonFeeder: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
try:
    if 'datasets_loaded' in locals():
        print("Creating DataLoaders...\n")
        
        # Create DataLoaders
        train_loader = torch.utils.data.DataLoader(
            datasets_loaded['train'],
            batch_size=4,
            shuffle=True,
            num_workers=0,  # Set to 0 for testing
        )
        
        dev_loader = torch.utils.data.DataLoader(
            datasets_loaded['dev'],
            batch_size=4,
            shuffle=False,
            num_workers=0,
        )
        
        print(f"✓ DataLoaders created")
        print(f"  Train batches: {len(train_loader)}")
        print(f"  Dev batches:   {len(dev_loader)}\n")
        
        # Try loading one batch
        print("Loading sample batch from train...")
        for batch_idx, batch in enumerate(train_loader):
            print(f"  Batch {batch_idx}:")
            print(f"    Keys: {batch.keys() if isinstance(batch, dict) else type(batch)}")
            if isinstance(batch, dict):
                for key, val in batch.items():
                    if isinstance(val, torch.Tensor):
                        print(f"      {key}: {val.shape}")
                    else:
                        print(f"      {key}: {type(val)} (len={len(val) if hasattr(val, '__len__') else 'N/A'})")
            print(f"  ✓ Sample batch loaded successfully\n")
            break  # Only load one batch
        
except Exception as e:
    print(f"❌ Error with DataLoader: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
print("\n" + "=" * 60)
print("📋 FINAL SUMMARY")
print("=" * 60)

summary_data = {
    'Metric': [
        'CSV Train',
        'CSV Dev',
        'CSV Test',
        'CSV Total',
        '',
        'SkeletonFeeder Train',
        'SkeletonFeeder Dev',
        'SkeletonFeeder Test',
        'SkeletonFeeder Total'
    ],
    'Count': [
        train_count,
        dev_count,
        test_count,
        train_count + dev_count + test_count,
        '',
        len(datasets_loaded['train']) if 'datasets_loaded' in locals() else 'N/A',
        len(datasets_loaded['dev']) if 'datasets_loaded' in locals() else 'N/A',
        len(datasets_loaded['test']) if 'datasets_loaded' in locals() else 'N/A',
        (len(datasets_loaded['train']) + len(datasets_loaded['dev']) + len(datasets_loaded['test'])) if 'datasets_loaded' in locals() else 'N/A'
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("=" * 60)

# Sanity checks
print("\n✅ SANITY CHECKS:")
checks = [
    ("CSV data loaded", train_count > 0 and dev_count > 0 and test_count > 0),
    ("SkeletonFeeder works", 'datasets_loaded' in locals() and all(len(datasets_loaded[s]) > 0 for s in ['train', 'dev', 'test'])),
    ("SkeletonFeeder train > 0", 'datasets_loaded' in locals() and len(datasets_loaded['train']) > 0),
    ("SkeletonFeeder dev > 0", 'datasets_loaded' in locals() and len(datasets_loaded['dev']) > 0),
    ("SkeletonFeeder test > 0", 'datasets_loaded' in locals() and len(datasets_loaded['test']) > 0),
]

for check_name, result in checks:
    status = "✓" if result else "✗"
    print(f"  {status} {check_name}")

print("\n✅ All checks passed!" if all(r for _, r in checks) else "\n⚠️  Some checks failed!")

In [ ]:
print("=" * 60)
print("📊 REPETITION ANALYSIS (R01-R05)")
print("=" * 60 + "\n")

def extract_repetition(video_id):
    """Extract repetition number from video_id (e.g., P01_S001_R01 -> R01)"""
    parts = video_id.split('_')
    if len(parts) >= 3:
        return parts[-1]  # R01, R02, R03, R04, R05
    return None

# Analyze CSV data
print("📝 CSV FILE DISTRIBUTION:\n")
if train_df is not None:
    train_reps = train_df['id'].apply(extract_repetition).value_counts().sort_index()
    print(f"Train CSV repetitions:")
    print(train_reps)
    print()

if dev_df is not None:
    dev_reps = dev_df['id'].apply(extract_repetition).value_counts().sort_index()
    print(f"Dev CSV repetitions:")
    print(dev_reps)
    print()

if test_df is not None:
    test_reps = test_df['id'].apply(extract_repetition).value_counts().sort_index()
    print(f"Test CSV repetitions:")
    print(test_reps)
    print()

# Analyze SkeletonFeeder data
print("\n🔄 SKELETON FEEDER REPETITION DISTRIBUTION:\n")
if 'datasets_loaded' in locals():
    for split in ['train', 'dev', 'test']:
        try:
            video_ids = [item['video_id'] for item in datasets_loaded[split].inputs_list]
            reps = pd.Series([extract_repetition(vid) for vid in video_ids]).value_counts().sort_index()
            print(f"{split.upper()} SkeletonFeeder repetitions:")
            print(reps)
            print()
        except Exception as e:
            print(f"Error analyzing {split}: {e}\n")

print("=" * 60)
print("💡 INSIGHT: Jika ada perbedaan jumlah per repetisi antara CSV dan")
print("   SkeletonFeeder, berarti skeleton pose untuk repetisi tersebut")
print("   tidak tersedia atau gagal di-ekstraksi.")
print("=" * 60 + "\n")

## 6.5 Analyze Repetition Distribution (R01-R05)

## 8. How to Run This Notebook

**In VS Code:**
1. Open this notebook file in VS Code
2. Click "Run All" button or run each cell individually (Shift+Enter)
3. Check the Output panel to see the dataset loading results
4. The summary at the end will show:
   - How many samples from CSV files
   - How many samples actually loaded by SkeletonFeeder
   - Whether all sanity checks pass

**Expected Output:**
- Train: ~359 samples (actual may vary based on available skeleton files)
- Dev: ~119 samples
- Test: ~29 samples

**If counts are lower than expected:**
- Check that skeleton files exist in `/data/kota502/github_repo/MahardikaPratama/dataset/mslr_dataset2/`
- Verify the dataset path configuration in `configs/dataset_configs/bisindo.yaml`
- Check for missing or corrupted skeleton files in the dataset

## 7. Summary & Sanity Checks

## 6. Test DataLoader with Batch Loading

## 5. Load with SkeletonFeeder (Actual Dataset Class)

## 4. Inspect Sample Data

## 3. Load Dataset from CSV

## 2. Set Paths & Configuration

## 1. Import Required Libraries